# DeepTrace — Phases 3 & 4: train and evaluate a model

**Run on:** Kaggle (free) · GPU **T4 x2 or P100** · Internet **On**
**Inputs:** `deeptrace-processed` (output of `01_data_prep_and_audit`).

Run this notebook once per model: `effnet_b0` (baseline), `convnext_tiny`, `clip_probe`, and optionally
`vit_small`. Use **Save Version → Save & Run All** so it runs in the background and keeps the outputs.

1. **Train** on the 140k `train` split, early-stopping on `val_select` ROC-AUC (CLIP: fit a linear probe).
2. **Predict** on `val_calib`, `test` and (if present) the cross-generator set.
3. **Calibrate** temperature, threshold and uncertainty band on `val_calib` only.
4. **Evaluate** every test set with bootstrap 95% CIs.

Seeds: first run `SEEDS = [42]` for every model. For the finalist(s), rerun with `[42, 43, 44]` and report
mean ± std across seeds. Evaluation below uses the first seed's checkpoint.

In [ ]:
# ---- Setup: clone your repo + install the few extras Kaggle doesn't ship ----
import os, subprocess, sys
from pathlib import Path

REPO_URL = "https://github.com/Vuday3336/DeepTrace-AI-face-detector.git"
ON_KAGGLE = Path("/kaggle/working").exists()
WORK = Path("/kaggle/working") if ON_KAGGLE else Path("/content")
REPO = WORK / "deeptrace"
ML = REPO / "ml"

if not REPO.exists():
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(REPO)], check=True)
os.chdir(ML)
print("Working in", Path.cwd())

subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements/kaggle.txt", "-r", "requirements/train.txt"], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements/facenet.txt", "--no-deps"], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", ".", "--no-deps"], check=True)
os.environ["NO_ALBUMENTATIONS_UPDATE"] = "1"

import torch
print("torch", torch.__version__, "| GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none")

def run(*args):
    print("$", " ".join(map(str, args)))
    subprocess.run([sys.executable, *map(str, args)], check=True)

In [ ]:
# ---- Locate Phase 2 outputs among the attached inputs ----
import os, json, shutil
from pathlib import Path

def find_dir(root, predicate, max_depth=6):
    base = len(Path(root).parts)
    for dirpath, dirnames, _ in os.walk(root):
        p = Path(dirpath)
        if len(p.parts) - base >= max_depth:
            dirnames.clear(); continue
        if predicate(p):
            return p
    return None

INPUT = Path("/kaggle/input") if ON_KAGGLE else Path("/content/input")
PROCESSED = find_dir(INPUT, lambda p: p.name == "processed" and (p / "140k").is_dir())
MANIFESTS = find_dir(INPUT, lambda p: (p / "140k_processed.csv.gz").is_file())
if PROCESSED is None or MANIFESTS is None:
    raise FileNotFoundError("Attach the 'deeptrace-processed' dataset (output of 01_data_prep_and_audit).")
RAW_140K = find_dir(INPUT, lambda p: p.name == "real-vs-fake" and (p / "test").is_dir())
RAW_CROSSGEN = find_dir(INPUT, lambda p: p.name == "crossgen" and (p / "fake").is_dir())
PROCESSED_MANIFESTS = [str(m) for m in (MANIFESTS / "140k_processed.csv.gz", MANIFESTS / "crossgen_processed.csv.gz") if m.is_file()]
print("processed crops:", PROCESSED)
print("manifests:", PROCESSED_MANIFESTS)
print("raw 140k:", RAW_140K, "| raw crossgen:", RAW_CROSSGEN)

In [ ]:
MODEL = "effnet_b0"          # effnet_b0 | convnext_tiny | vit_small | clip_probe
SEEDS = [42]
RUNS = WORK / "runs"
REPORTS = ML / "reports"

## 1. Train

In [ ]:
for seed in SEEDS:
    run_dir = RUNS / f"{MODEL}_seed{seed}"
    if MODEL == "clip_probe":
        run("scripts/train_clip_probe.py", "--manifest", MANIFESTS / "140k_processed.csv.gz",
            "--data-root", PROCESSED, "--run-dir", run_dir, "--seed", seed)
    else:
        run("scripts/train.py", "--config", f"configs/train/{MODEL}.yaml",
            "--manifest", MANIFESTS / "140k_processed.csv.gz", "--data-root", PROCESSED,
            "--run-dir", run_dir, "--seed", seed)
    print(json.load(open(run_dir / "final_metrics.json")))
CHECKPOINT = RUNS / f"{MODEL}_seed{SEEDS[0]}" / "best.pt"

In [ ]:
# Learning curves (fine-tuned models only)
import pandas as pd
import matplotlib.pyplot as plt
history_file = RUNS / f"{MODEL}_seed{SEEDS[0]}" / "metrics_epoch.jsonl"
if history_file.exists():
    h = pd.read_json(history_file, lines=True)
    fig, ax = plt.subplots(1, 2, figsize=(10, 3.5))
    h.plot(x="epoch", y=["train_loss", "val_loss"], ax=ax[0], marker="o"); ax[0].set_title("loss")
    h.plot(x="epoch", y="val_auc", ax=ax[1], marker="o"); ax[1].set_title("val_select ROC-AUC")
    plt.show()

## 2–4. Predict → calibrate (val_calib only) → evaluate

In [ ]:
PRED = REPORTS / f"predictions_{MODEL}.csv"
manifest_args = [a for m in PROCESSED_MANIFESTS for a in ("--manifest", m)]
run("scripts/predict.py", "--checkpoint", CHECKPOINT, *manifest_args, "--data-root", PROCESSED,
    "--splits", "val_calib", "test", "crossgen", "own", "--out", PRED)
run("scripts/calibrate.py", "--predictions", PRED, "--model-name", MODEL)
run("scripts/evaluate.py", "--predictions", PRED, "--calibration", REPORTS / f"calibration_{MODEL}.json",
    "--model-name", MODEL)

In [ ]:
report = json.load(open(REPORTS / f"eval_{MODEL}.json"))
rows = []
for name, m in report["test_sets"].items():
    t = m["at_tuned_threshold"]
    ci = m["ci95"]["roc_auc"]
    rows.append({"test_set": name, "n": m["n"], "roc_auc": m["roc_auc"],
                 "auc_ci95": f"[{ci['low']:.3f}, {ci['high']:.3f}]" if ci else None,
                 "bal_acc": t["balanced_accuracy"], "fpr": t["fpr"], "tpr": t["recall_tpr"], "ece": m["ece_15"]})
pd.DataFrame(rows).round(4)

## Save

When the committed run finishes: **Output → New Dataset** named `deeptrace-runs-<model>`. Notebook
`03_robustness_explain_export` attaches these. Also copy `ml/reports/*.json` and `docs/plots/eval/` into
your repo — they are the only source of the numbers in the README.

In [ ]:
bundle = WORK / f"deeptrace_{MODEL}_artifacts"
shutil.rmtree(bundle, ignore_errors=True)
shutil.copytree(REPORTS, bundle / "ml/reports")
shutil.copytree(REPO / "docs/plots", bundle / "docs/plots", dirs_exist_ok=True)
print(shutil.make_archive(str(bundle), "zip", bundle))